---
title: "Aftyn Behn retrospective"
description: "What happened in the TN special election and what does it mean for 2026 and 2028?"
author: "chicken"
date: "1/01/2026"
categories:
    - "politics"
image: charts/5v5_xgf_xga_NSH.png
lightbox: true
draft: false
---

### Import dependencies and set options

In [ ]:
# chickenstats library and utilities
from chickenstats.utilities import ChickenProgress
from chickenstats.chicken_nhl._helpers import charts_directory

# data science libraries
import pandas as pd
import geopandas as gpd
import polars as pl
import polars_st as pl_st

# plotting libraries
import matplotlib.pyplot as plt

# miscellaneous libraries
from dotenv import load_dotenv
import os

# census API libraries
from census import Census
from us import states

In [ ]:
# set pandas options so that there are no columns abbreviated
pd.set_option("display.max_columns", None)
pl.Config.set_tbl_cols(-1)

polars.config.Config

### Create directory for charts

In [ ]:
charts_directory()

### chickenstats matplotlib styles

In [ ]:
plt.style.use("chickenstats")  # this is available when you import chickenstats.utilities

### Getting and setting environment variables

In [ ]:
load_dotenv()
census_key = os.environ.get("CENSUS_KEY")

### Variables for Census API

In [ ]:
basic_cols = {
    "NAME": "tract_name",
    #'AIANHH': 'AIANHH',
    #'AIARO': 'AIARO',
    #'AIHHTL': 'AIHHTL',
    #'AIRES': 'AIRES',
    #'ANRC': 'ANRC',
    "B01003_001E": "total_population",
}

sex_cols = {
    "B01001_001E": "sex_total",
    "B01001_002E": "sex_male",
    "B01001_003E": "sex_male_5_younger",
    "B01001_004E": "sex_male_5_9",
    "B01001_005E": "sex_male_10_14",
    "B01001_006E": "sex_male_15_17",
    "B01001_007E": "sex_male_18_19",
    "B01001_008E": "sex_male_20",
    "B01001_009E": "sex_male_21",
    "B01001_010E": "sex_male_22_24",
    "B01001_011E": "sex_male_25_29",
    "B01001_012E": "sex_male_30_34",
    "B01001_013E": "sex_male_35_39",
    "B01001_014E": "sex_male_40_44",
    "B01001_015E": "sex_male_45_49",
    "B01001_016E": "sex_male_50_54",
    "B01001_017E": "sex_male_55_59",
    "B01001_018E": "sex_male_60_61",
    "B01001_019E": "sex_male_62_64",
    "B01001_020E": "sex_male_65_66",
    "B01001_021E": "sex_male_67_69",
    "B01001_022E": "sex_male_70_74",
    "B01001_023E": "sex_male_75_79",
    "B01001_024E": "sex_male_80_84",
    "B01001_025E": "sex_male_85_plus",
    "B01001_026E": "sex_female",
    "B01001_027E": "sex_female_5_younger",
    "B01001_028E": "sex_female_5_9",
    "B01001_029E": "sex_female_10_14",
    "B01001_030E": "sex_female_15_17",
    "B01001_031E": "sex_female_18_19",
    "B01001_032E": "sex_female_20",
    "B01001_033E": "sex_female_21",
    "B01001_034E": "sex_female_22_24",
    "B01001_035E": "sex_female_25_29",
    "B01001_036E": "sex_female_30_34",
    "B01001_037E": "sex_female_35_39",
    "B01001_038E": "sex_female_40_44",
    "B01001_039E": "sex_female_45_49",
    "B01001_040E": "sex_female_50_54",
    "B01001_041E": "sex_female_55_59",
    "B01001_042E": "sex_female_60_61",
    "B01001_043E": "sex_female_62_64",
    "B01001_044E": "sex_female_65_66",
    "B01001_045E": "sex_female_67_69",
    "B01001_046E": "sex_female_70_74",
    "B01001_047E": "sex_female_75_79",
    "B01001_048E": "sex_female_80_84",
    "B01001_049E": "sex_female_85_plus",
}

education_cols = {
    "B15003_001E": "education_total",
    "B15003_002E": "education_no_schooling",
    "B15003_017E": "education_hs_diploma",
    "B15003_018E": "education_ged",
    "B15003_022E": "education_bachelors",
    "B15003_023E": "education_masters",
    "B15003_024E": "education_professional",
    "B15003_025E": "education_doctorate",
}

education_sex_cols = {
    "B15002_001E": "education_sex_total",
    "B15002_002E": "education_male_total",
    "B15002_003E": "education_male_no_schooling",
    "B15002_011E": "education_male_hs_grad",
    "B15002_015E": "education_male_bachelors",
    "B15002_016E": "education_male_masters",
    "B15002_017E": "education_male_professional",
    "B15002_018E": "education_male_doctorate",
    "B15002_019E": "education_female_total",
    "B15002_020E": "education_female_no_schooling",
    "B15002_028E": "education_female_hs_grad",
    "B15002_032E": "education_female_bachelors",
    "B15002_033E": "education_female_masters",
    "B15002_034E": "education_female_professional",
    "B15002_035E": "education_female_doctorate",
}

couple_type_cols = {
    "B11009_001E": "couples_total",
    "B11009_002E": "couples_married",
    "B11009_003E": "couples_married_opp_sex",
    "B11009_004E": "couples_married_same_sex",
    "B11009_005E": "couples_married_male_male",
    "B11009_006E": "couples_married_female_female",
    "B11009_007E": "couples_cohabit",
    "B11009_008E": "couples_cohabit_opp_sex",
    "B11009_009E": "couples_cohabit_same_sex",
    "B11009_010E": "couples_cohabit_male_male",
    "B11009_011E": "couples_cohabit_female_female",
    "B11009_012E": "couples_all_others",
}

household_type_cols = {
    "B11012_001E": "households_total",
    "B11012_002E": "households_married",
    "B11012_003E": "households_married_children",
    "B11012_004E": "households_married_no_children",
    "B11012_005E": "households_cohabit",
    "B11012_006E": "households_cohabit_children",
    "B11012_007E": "households_cohabit_no_children",
    "B11012_008E": "households_female_single",
    "B11012_009E": "households_female_single_alone",
    "B11012_010E": "households_female_single_children",
    "B11012_011E": "households_female_single_relatives_no_children",
    "B11012_012E": "households_female_single_non_relatives_only",
    "B11012_013E": "households_male_single",
    "B11012_014E": "households_male_single_alone",
    "B11012_015E": "households_male_single_children",
    "B11012_016E": "households_male_single_relatives_no_children",
    "B11012_017E": "households_male_single_non_relatives_only",
}

household_type_income_cols = {
    "B19131_001E": "hh_income_hh_type_total",
    "B19131_057E": "hh_income_male_single_no_children_10_less",
    "B19131_058E": "hh_income_male_single_no_children_10_15",
    "B19131_059E": "hh_income_male_single_no_children_15_19",
    "B19131_060E": "hh_income_male_single_no_children_20_24",
    "B19131_061E": "hh_income_male_single_no_children_25_29",
    "B19131_062E": "hh_income_male_single_no_children_30_34",
    "B19131_063E": "hh_income_male_single_no_children_35_39",
    "B19131_064E": "hh_income_male_single_no_children_40_44",
    "B19131_065E": "hh_income_male_single_no_children_45_49",
    "B19131_066E": "hh_income_male_single_no_children_50_59",
    "B19131_067E": "hh_income_male_single_no_children_60_74",
    "B19131_068E": "hh_income_male_single_no_children_75_99",
    "B19131_069E": "hh_income_male_single_no_children_100_124",
    "B19131_070E": "hh_income_male_single_no_children_125_149",
    "B19131_071E": "hh_income_male_single_no_children_150_199",
    "B19131_072E": "hh_income_male_single_no_children_200_plus",
    "B19131_092E": "hh_income_female_single_no_children_10_less",
    "B19131_093E": "hh_income_female_single_no_children_10_15",
    "B19131_094E": "hh_income_female_single_no_children_15_19",
    "B19131_095E": "hh_income_female_single_no_children_20_24",
    "B19131_096E": "hh_income_female_single_no_children_25_29",
    "B19131_097E": "hh_income_female_single_no_children_30_34",
    "B19131_098E": "hh_income_female_single_no_children_35_39",
    "B19131_099E": "hh_income_female_single_no_children_40_44",
    "B19131_100E": "hh_income_female_single_no_children_45_49",
    "B19131_101E": "hh_income_female_single_no_children_50_59",
    "B19131_102E": "hh_income_female_single_no_children_60_74",
    "B19131_103E": "hh_income_female_single_no_children_75_99",
    "B19131_104E": "hh_income_female_single_no_children_100_124",
    "B19131_105E": "hh_income_female_single_no_children_125_149",
    "B19131_106E": "hh_income_female_single_no_children_150_199",
    "B19131_107E": "hh_income_female_single_no_children_200_plus",
}

poverty_cols = {
    "C17002_001E": "income_to_poverty_total",
    "C17002_002E": "income_to_poverty_0_50",
    "C17002_003E": "income_to_poverty_51_99",
    "C17002_004E": "income_to_poverty_100_124",
    "C17002_005E": "income_to_poverty_125_149",
    "C17002_006E": "income_to_poverty_150_184",
    "C17002_007E": "income_to_poverty_185_199",
    "C17002_008E": "income_to_poverty_200",
}

income_cols = {
    "B19001_001E": "hh_income_total",
    "B19001_002E": "hh_income_10_less",
    "B19001_003E": "hh_income_10_15",
    "B19001_004E": "hh_income_15_20",
    "B19001_005E": "hh_income_20_25",
    "B19001_006E": "hh_income_25_30",
    "B19001_007E": "hh_income_30_35",
    "B19001_008E": "hh_income_35_40",
    "B19001_009E": "hh_income_40_45",
    "B19001_010E": "hh_income_45_50",
    "B19001_011E": "hh_income_50_60",
    "B19001_012E": "hh_income_60_75",
    "B19001_013E": "hh_income_75_100",
    "B19001_014E": "hh_income_100_125",
    "B19001_015E": "hh_income_125_150",
    "B19001_016E": "hh_income_150_200",
    "B19001_017E": "hh_income_200_plus",
}

employment_cols = {
    "B23025_001E": "employment_total",
    "B23025_002E": "employment_in_labor_force",
    "B23025_003E": "employment_civilian_total",
    "B23025_004E": "employment_civilian_employed",
    "B23025_005E": "employment_civilian_unemployed",
    "B23025_006E": "employment_armed_forces",
    "B23025_007E": "employment_not_in_labor_force",
}

commute_cols = {
    "B08303_001E": "travel_time_total",
    "B08303_002E": "travel_time_5_less",
    "B08303_003E": "travel_time_5_9",
    "B08303_004E": "travel_time_10_14",
    "B08303_005E": "travel_time_15_19",
    "B08303_006E": "travel_time_20_24",
    "B08303_007E": "travel_time_25_29",
    "B08303_008E": "travel_time_30_34",
    "B08303_009E": "travel_time_35_39",
    "B08303_010E": "travel_time_40_44",
    "B08303_011E": "travel_time_45_59",
    "B08303_012E": "travel_time_60_89",
    "B08303_013E": "travel_time_90_plus",
}

race_cols = {
    "B02001_001E": "race_total",
    "B02001_002E": "race_white",
    "B02001_003E": "race_black",
    "B02001_004E": "race_indigenous",
    "B02001_005E": "race_asian",
    "B02001_006E": "race_hawaiian_pi",
    "B02001_007E": "race_two",
}

citizenship_cols = {
    "B05001_001E": "citizenship_total",
    "B05001_002E": "citizenship_us_us",
    "B05001_003E": "citizenship_us_islands",
    "B05001_004E": "citizenship_us_abroad",
    "B05001_005E": "citizenship_us_naturalized",
    "B05001_006E": "citizenship_not_us",
}

nativity_cols = {
    "B05012_001E": "nativity_total",
    "B05012_002E": "nativity_us_born",
    "B05012_003E": "nativity_foreign_born",
}

internet_cols = {
    "B28002_001E": "internet_total",
    "B28002_002E": "internet_total_with_internet",
    "B28002_003E": "internet_dial_up",
    "B28002_004E": "internet_broadband",
    "B28002_005E": "internet_cellular",
    "B28002_006E": "internet_cellular_only",
    "B28002_007E": "internet_broadband_dsl",
    "B28002_008E": "internet_broadband_dsl_only",
    "B28002_009E": "internet_satellite",
    "B28002_010E": "internet_satellite_only",
    "B28002_011E": "internet_other",
    "B28002_012E": "internet_access_no_subscription",
    "B28002_013E": "internet_no_access",
}

computer_cols = {
    "B28001_001E": "computers_total",
    "B28001_002E": "computers_one_or_more",
    "B28001_003E": "computers_desktop",
    "B28001_004E": "computers_desktop_only",
    "B28001_005E": "computers_smartphone",
    "B28001_006E": "computers_smartphone_only",
    "B28001_007E": "computers_tablet",
    "B28001_008E": "computers_tablet_only",
    "B28001_009E": "computers_other",
    "B28001_010E": "computers_other_only",
    "B28001_011E": "computers_no_computer",
}

computer_internet_cols = {}

vacancy_cols = {
    "B25004_001E": "vacancy_total",
    "B25004_002E": "vacancy_for_rent",
    "B25004_003E": "vacancy_rented_not_occupied",
    "B25004_004E": "vacancy_for_sale_only",
    "B25004_005E": "vacancy_sold_not_occupied",
    "B25004_006E": "vacancy_season_recreational",
    "B25004_007E": "vacancy_migrant_workies",
    "B25004_008E": "vacancy_other_vacant",
}

housing_cols = {
    "B25001_001E": "housing_total",
    "B25002_002E": "housing_occupied",
    "B25002_003E": "housing_vacant",
    "B25003_002E": "housing_owner_occupied",
    "B25003_003E": "housing_renter_occupied",
}

rent_cols = {
    "B25031_001E": "median_gross_rent",
    "B25031_002E": "median_gross_rent_no_bed",
    "B25031_003E": "median_gross_rent_1_bed",
    "B25031_004E": "median_gross_rent_2_bed",
    "B25031_005E": "median_gross_rent_3_bed",
    "B25031_006E": "median_gross_rent_4_bed",
    "B25031_007E": "median_gross_rent_5_bed",
}

housing_unit_cols = {
    "B25032_001E": "housing_units_total",
    "B25032_002E": "housing_units_owner_occupied_total",
    "B25032_003E": "housing_units_owner_occupied_1_detached",
    "B25032_004E": "housing_units_owner_occupied_1_attached",
    "B25032_005E": "housing_units_owner_occupied_2",
    "B25032_006E": "housing_units_owner_occupied_3_4",
    "B25032_007E": "housing_units_owner_occupied_5_9",
    "B25032_008E": "housing_units_owner_occupied_10_19",
    "B25032_009E": "housing_units_owner_occupied_20_49",
    "B25032_010E": "housing_units_owner_occupied_50_plus",
    "B25032_011E": "housing_units_owner_occupied_mobile_home",
    "B25032_012E": "housing_units_owner_occupied_boat_rv",
    "B25032_013E": "housing_units_renter_occupied_total",
    "B25032_014E": "housing_units_renter_occupied_1_detached",
    "B25032_015E": "housing_units_renter_occupied_1_attached",
    "B25032_016E": "housing_units_renter_occupied_2",
    "B25032_017E": "housing_units_renter_occupied_3_4",
    "B25032_018E": "housing_units_renter_occupied_5_9",
    "B25032_019E": "housing_units_renter_occupied_10_19",
    "B25032_020E": "housing_units_renter_occupied_20_49",
    "B25032_021E": "housing_units_renter_occupied_50_plus",
    "B25032_022E": "housing_units_renter_occupied_mobile_home",
    "B25032_023E": "housing_units_renter_occupied_boat_rv",
}

structure_built_cols = {
    "B25037_001E": "structure_median_year_total",
    "B25037_002E": "structure_median_year_owner_occupied",
    "B25037_003E": "structure_median_year_renter_occupied",
    "B25034_001E": "structure_total",
    "B25034_002E": "structure_2014_later",
    "B25034_003E": "structure_2010_2013",
    "B25034_004E": "structure_2000_2009",
    "B25034_005E": "structure_1990_1999",
    "B25034_006E": "structure_1980_1989",
    "B25034_007E": "structure_1970_1979",
    "B25034_008E": "structure_1960_1969",
    "B25034_009E": "structure_1950_1959",
    "B25034_010E": "structure_1940_1949",
    "B25034_011E": "structure_1939_earlier",
}

mobility_cols = {
    "B25039_001E": "median_year_moved_total",
    "B25039_002E": "median_year_moved_owner_occupied",
    "B25039_003E": "median_year_moved_renter_occupied",
}

housing_value_cols = {
    "B25076_001E": "housing_value_lower_quartile",
    "B25077_001E": "housing_value_median",
    "B25078_001E": "housing_value_upper_quartile",
    "B25075_001E": "housing_value_total",
    "B25075_002E": "housing_value_10000_less",
    "B25075_003E": "housing_value_10000_14999",
    "B25075_004E": "housing_value_15000_19999",
    "B25075_005E": "housing_value_20000_24999",
    "B25075_006E": "housing_value_25000_29999",
    "B25075_007E": "housing_value_30000_34999",
    "B25075_008E": "housing_value_35000_39999",
    "B25075_009E": "housing_value_40000_49999",
    "B25075_010E": "housing_value_50000_59999",
    "B25075_011E": "housing_value_60000_69000",
    "B25075_012E": "housing_value_70000_79000",
    "B25075_013E": "housing_value_80000_89000",
    "B25075_014E": "housing_value_90000_99999",
    "B25075_015E": "housing_value_100000_124999",
    "B25075_016E": "housing_value_125000_149999",
    "B25075_017E": "housing_value_150000_174999",
    "B25075_018E": "housing_value_175000_199999",
    "B25075_019E": "housing_value_200000_249999",
    "B25075_020E": "housing_value_250000_299999",
    "B25075_021E": "housing_value_300000_399999",
    "B25075_022E": "housing_value_400000_499999",
    "B25075_023E": "housing_value_500000_749999",
    "B25075_024E": "housing_value_750000_999999",
    "B25075_025E": "housing_value_1000000_1499999",
    "B25075_026E": "housing_value_1500000_1999999",
    "B25075_027E": "housing_value_2000000_plus",
}

### Function to scrape census data

In [ ]:
def census_data(years: list[str | int], state_abbreviation: str = "TN") -> pl.DataFrame:
    """Docstring."""
    with ChickenProgress() as progress:
        census_list = []
        tract_list = []

        pbar_message = "Downloading census data..."
        progress_total = len(years)
        progress_task = progress.add_task(pbar_message, total=progress_total, refresh=True)
        progress.start_task(progress_task)

        state_fips = states.lookup(state_abbreviation).fips

        for year in years:
            api_dict = {
                **basic_cols,
                **sex_cols,
                **education_cols,
                **education_sex_cols,
                **couple_type_cols,
                **household_type_cols,
                **household_type_income_cols,
                **income_cols,
                **poverty_cols,
                **citizenship_cols,
                **nativity_cols,
                **race_cols,
                **employment_cols,
                **commute_cols,
                **internet_cols,
                **computer_cols,
                **vacancy_cols,
                **housing_cols,
                **housing_value_cols,
                **rent_cols,
                **housing_unit_cols,
                **structure_built_cols,
                **mobility_cols,
            }

            fields = list(api_dict.keys())

            response = c.acs5.state_county_tract(
                fields=fields, state_fips=state_fips, county_fips="*", tract="*", year=year
            )

            df = (
                pl.DataFrame(response)
                .rename(api_dict)
                .with_columns(year=year, geo_id=pl.col("state") + pl.col("county") + pl.col("tract"))
            )

            census_list.append(df)

            tract_url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"

            tract_df = gpd.read_file(tract_url)  # .with_columns(year=year)

            tract_list.append(tract_df)

            progress.update(progress_task, description=pbar_message, advance=1, refresh=True)

    census = pl.concat(census_list)
    tract = pd.concat(tract_list)

    return census, tract

In [ ]:
c = Census(census_key)

In [ ]:
test = census_data([2020])

Output()

In [ ]:
test[1]  # .with_columns(geometry=pl.col("geometry").bin.encode("hex"),
# geometry2=pl_st.polygon("geometry"),
#  )

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,47,175,925200,47175925200,9252,Census Tract 9252,G5020,S,227429512,1667739,+35.7438100,-085.4940227,"POLYGON ((-85.61516 35.76106, -85.61509 35.761..."
1,47,175,925000,47175925000,9250,Census Tract 9250,G5020,S,480712883,1225717,+35.6695378,-085.4220628,"POLYGON ((-85.60513 35.70854, -85.60511 35.708..."
2,47,003,950201,47003950201,9502.01,Census Tract 9502.01,G5020,S,121774227,0,+35.6517480,-086.5575518,"POLYGON ((-86.64406 35.64029, -86.64375 35.642..."
3,47,003,950202,47003950202,9502.02,Census Tract 9502.02,G5020,S,110617191,700793,+35.5845755,-086.5790796,"POLYGON ((-86.66377 35.58189, -86.66367 35.582..."
4,47,093,003300,47093003300,33,Census Tract 33,G5020,S,5860088,229299,+36.0020586,-083.8371218,"POLYGON ((-83.86208 35.99255, -83.86207 35.992..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1696,47,073,050501,47073050501,505.01,Census Tract 505.01,G5020,S,83355996,0,+36.5659159,-082.7715246,"POLYGON ((-82.85011 36.54107, -82.84955 36.541..."
1697,47,073,050601,47073050601,506.01,Census Tract 506.01,G5020,S,23847669,177932,+36.5836443,-082.6318663,"POLYGON ((-82.68939 36.58793, -82.68936 36.588..."
1698,47,073,050602,47073050602,506.02,Census Tract 506.02,G5020,S,23386546,668716,+36.5467218,-082.6458382,"POLYGON ((-82.69446 36.54856, -82.69445 36.548..."
1699,47,073,050502,47073050502,505.02,Census Tract 505.02,G5020,S,32873931,311980,+36.5332207,-082.7586081,"POLYGON ((-82.82242 36.52006, -82.82171 36.520..."


In [ ]:
year = 2020
state_abbreviation = "TN"
state_fips = states.lookup(state_abbreviation).fips

tract_url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"

gpd_df = gpd.read_file(tract_url)  # .with_columns(year=year)

In [ ]:
pl_df = pl_st.read_file(tract_url)

In [ ]:
pl_df.with_columns(geometry2=pl.col("geometry").bin.reinterpret(dtype=pl.Float32, endianness="big"))

STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,geometry2
str,str,str,str,str,str,str,str,i64,i64,str,str,binary,f32
"""47""","""175""","""925200""","""47175925200""","""9252""","""Census Tract 9252""","""G5020""","""S""",227429512,1667739,"""+35.7438100""","""-085.4940227""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00\xf1\x0b\x00\x00Lk\xd3\xd8^gU\xc0\xdb\x14\x8f\x8bj\xe1A@G\xc8@\x9e]gU\xc0\xb4:9Cq\xe1A@\x89\x05\xbe\xa2[gU\xc0W\x1f\x0f""…",null
"""47""","""175""","""925000""","""47175925000""","""9250""","""Census Tract 9250""","""G5020""","""S""",480712883,1225717,"""+35.6695378""","""-085.4220628""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00\xe8\x0f\x00\x00\x06\xf5-s\xbafU\xc0\xd5\xeb\x16\x81\xb1\xdaA@A'\x84\x0e\xbafU\xc0\x81]M\x9e\xb2\xdaA@\xe8\xf5'\xf1\xb9fU\xc0E+\xf7""…",null
"""47""","""003""","""950201""","""47003950201""","""9502.01""","""Census Tract 9502.01""","""G5020""","""S""",121774227,0,"""+35.6517480""","""-086.5575518""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00\xf9\x02\x00\x00\xb92\xa868\xa9U\xc0\xf9\xf1\x97\x16\xf5\xd1A@Ku\x01/3\xa9U\xc0\x9cP\x88\x80C\xd2A@\xe7\xfa>\x1c$\xa9U\xc0#d\x20""…",null
"""47""","""003""","""950202""","""47003950202""","""9502.02""","""Census Tract 9502.02""","""G5020""","""S""",110617191,700793,"""+35.5845755""","""-086.5790796""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00\xf6\x05\x00\x008\x84*5{\xaaU\xc0\xbbbFx{\xcaA@\xe5]\xf5\x80y\xaaU\xc0\xe2vhX\x8c\xcaA@\x8e\xc8w)u\xaaU\xc0\xc2/\xf5""…",null
"""47""","""093""","""003300""","""47093003300""","""33""","""Census Tract 33""","""G5020""","""S""",5860088,229299,"""+36.0020586""","""-083.8371218""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00\x04\x01\x00\x00\xd1W\x90f,\xf7T\xc0\xca\xc1l\x02\x0c\xffA@Nyt#,\xf7T\xc0u\xcb\x0e\xf1\x0f\xffA@ri\xfc\xc2+\xf7T\xc0\xa3\xe7\x16""…",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""47""","""073""","""050501""","""47073050501""","""505.01""","""Census Tract 505.01""","""G5020""","""S""",83355996,0,"""+36.5659159""","""-082.7715246""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00S\x05\x00\x00q\xc60'h\xb6T\xc0\xf0\xf8\xf6\xaeAEB@\xe7\x8b\xbd\x17_\xb6T\xc0e\xde\xaa\xebPEB@)\x95\xf0\x84^\xb6T\xc0\xeey\xfe""…",null
"""47""","""073""","""050601""","""47073050601""","""506.01""","""Census Tract 506.01""","""G5020""","""S""",23847669,177932,"""+36.5836443""","""-082.6318663""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00l\x02\x00\x00\xd8E\xd1\x03\x1f\xacT\xc0\xcc""\x14[AKB@\xa3\x04\xfd\x85\x1e\xacT\xc0\x9b\x02\x99\x9dEKB@8\x9c\xf9\xd5\x1c\xacT\xc0\xb6\x7fe""…",null
"""47""","""073""","""050602""","""47073050602""","""506.02""","""Census Tract 506.02""","""G5020""","""S""",23386546,668716,"""+36.5467218""","""-082.6458382""","b""\x01\x03\x00\x00\x20\xad\x10\x00\x00\x01\x00\x00\x00\xa7\x02\x00\x00lZ)\x04r\xacT\xc0%\x03@\x157FB@+k\x9b\xe2q\xacT\xc0\xc8\xb3\xcb\xb7>FB@\xa8\x8c\x7f\x9fq\xacT\xc0s\xf1\xb7""…",null
